In [ ]:
%load_ext autoreload
%autoreload 2
import os
import torch
import matplotlib.pyplot as plt
import numpy as np

from compactreasoningmodels.solvers import (
    MAC, GeneticAlgorithmDET, GeneticAlgorithmDEP,
    GDGlobalAdamSolver, GDGlobalSGDSolver, GDBangBangSolver,
    GDGaussSeidelAdamSolver, GDGaussSeidelSGDSolver, 
    GDJacobiAdamSolver, GDJacobiSGDSolver,
    ModelSolver, BacktrackingSearch
)
from compactreasoningmodels.datasets import NonogramDataset
from compactreasoningmodels.datasets.collate import collate_default
from compactreasoningmodels.utils.display import display_grid

if 'original_dir' not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, ".."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "models")


In [ ]:
dataset = NonogramDataset("raw/nonogram_5x5.jsonl", max_size=5)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_default) 
sample_clues, sample_grids, _, _ = next(iter(dataloader))
sample_clues = sample_clues.squeeze(0).reshape(2, 5, 3)
sample_grids = sample_grids.squeeze(0).reshape(5, 5)
fig, ax = plt.subplots(figsize=(5, 5))
display_grid(ax, sample_grids, clues=sample_clues, title='')

In [ ]:
def display_solution_trace(solver, clues, display_steps=10, sampling_ratio=0.5, wrap=5):
    width = min(display_steps, wrap)
    height = display_steps // width + (1 if display_steps % width > 0 else 0)
    fig, axes = plt.subplots(height, width, figsize=(5 * width, 5 * height))
    
    traces = solver.step(clues, num_steps=display_steps, sampling_ratio=sampling_ratio)[:display_steps]
    for i, ax in enumerate(axes.flatten()):
        display_grid(ax, traces[i], clues=clues, title=f'Step {i}')

In [ ]:
from compactreasoningmodels.losses.clue_reconstruction2 import ClueReconstructionLoss as ClueReconstructionLoss2
display_solution_trace(GDGlobalSGDSolver(loss_fn=ClueReconstructionLoss2(
                                            reduction="mean",
                                            aux_weight_start=0.2,
                                            aux_weight_end=0.5,
                                            entropy_weight_start=0.5,
                                            entropy_weight_end=0.2,
                                            anneal_steps=1500,
                                        )), sample_clues, display_steps=10, sampling_ratio=1.0, wrap=5)
display_solution_trace(GDGlobalSGDSolver(), sample_clues, display_steps=10, sampling_ratio=1.0, wrap=5)

In [ ]:
display_solution_trace(BacktrackingSearch(), sample_clues, display_steps=10, sampling_ratio=1.0, wrap=5)

In [ ]:
display_solution_trace(MAC(), sample_clues, display_steps=5, sampling_ratio=1.0, wrap=5)
display_solution_trace(MAC(), sample_clues, display_steps=10, sampling_ratio=0.5, wrap=5)
display_solution_trace(MAC(), sample_clues, display_steps=10, sampling_ratio=0.5, wrap=5)

In [ ]:
display_solution_trace(GeneticAlgorithmDET(), sample_clues, display_steps=10, sampling_ratio=1.0, wrap=5)
display_solution_trace(GeneticAlgorithmDET(), sample_clues, display_steps=10, sampling_ratio=0.5, wrap=5)
display_solution_trace(GeneticAlgorithmDEP(), sample_clues, display_steps=10, sampling_ratio=1.0, wrap=5)
display_solution_trace(GeneticAlgorithmDEP(), sample_clues, display_steps=10, sampling_ratio=0.5, wrap=5)


In [ ]:
display_solution_trace(GDGlobalAdamSolver(), sample_clues, display_steps=10, sampling_ratio=1.0)
display_solution_trace(GDGlobalAdamSolver(), sample_clues, display_steps=10, sampling_ratio=0.5)
display_solution_trace(GDGlobalSGDSolver(), sample_clues, display_steps=10, sampling_ratio=1.0)
display_solution_trace(GDGlobalSGDSolver(), sample_clues, display_steps=10, sampling_ratio=0.5)
display_solution_trace(GDBangBangSolver(), sample_clues, display_steps=10, sampling_ratio=1.0)
display_solution_trace(GDBangBangSolver(), sample_clues, display_steps=10, sampling_ratio=0.5)

In [ ]:
display_solution_trace(ModelSolver(), sample_clues, display_steps=10, sampling_ratio=1.0, wrap=5)
display_solution_trace(ModelSolver(), sample_clues, display_steps=10, sampling_ratio=0.9, wrap=5)
display_solution_trace(ModelSolver(), sample_clues, display_steps=10, sampling_ratio=0.5, wrap=5)
display_solution_trace(ModelSolver(), sample_clues, display_steps=10, sampling_ratio=0.5, wrap=5)